In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("irfanasrullah/groceries")

print("Path to dataset files:", path)

100%|██████████| 168k/168k [00:00<00:00, 696kB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/irfanasrullah/groceries/versions/2


In [ ]:
import os
print(os.listdir(path))

['groceries.csv', 'groceries - groceries.csv']


In [ ]:
df=pd.read_csv(path+'/groceries - groceries.csv')

In [ ]:
df.head()

,Item(s),Item 1,Item 2,Item 3,Item 4,Item 5,Item 6,Item 7,Item 8,Item 9,...,Item 23,Item 24,Item 25,Item 26,Item 27,Item 28,Item 29,Item 30,Item 31,Item 32
0,4,citrus fruit,semi-finished bread,margarine,ready soups,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,tropical fruit,yogurt,coffee,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,whole milk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,pip fruit,yogurt,cream cheese,meat spreads,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,other vegetables,whole milk,condensed milk,long life bakery product,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##Convert rows into transactions

In [ ]:
transactions = []

for i in range(len(df)):

    row = []

    for item in df.iloc[i]:

        if pd.notna(item):
            row.append(str(item))

    transactions.append(row)

print(transactions[:5])

[['citrus fruit', 'semi-finished bread', 'margarine', 'ready soups'], ['tropical fruit', 'yogurt', 'coffee'], ['whole milk'], ['pip fruit', 'yogurt', 'cream cheese', 'meat spreads'], ['other vegetables', 'whole milk', 'condensed milk', 'long life bakery product']]


In [ ]:
df = df.drop(columns=['Item(s)'])

In [ ]:
df.head()

,Item 1,Item 2,Item 3,Item 4,Item 5,Item 6,Item 7,Item 8,Item 9,Item 10,...,Item 23,Item 24,Item 25,Item 26,Item 27,Item 28,Item 29,Item 30,Item 31,Item 32
0,citrus fruit,semi-finished bread,margarine,ready soups,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,tropical fruit,yogurt,coffee,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,whole milk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,pip fruit,yogurt,cream cheese,meat spreads,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,other vegetables,whole milk,condensed milk,long life bakery product,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##Transaction Encoding

In [ ]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()

te_array = te.fit(transactions).transform(transactions)

basket = pd.DataFrame(
    te_array,
    columns=te.columns_
)

basket.head()

,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,baby food,bags,baking powder,bathroom cleaner,beef,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False


In [ ]:
from mlxtend.frequent_patterns import apriori
frequent_items=apriori(basket,min_support=0.02,use_colnames=True)
frequent_items.head()
#it works in descending order
#UHT-milk      3.35%
#Beef          5.25%
#Bottled beer  8.05%

,support,itemsets
0,0.033452,(UHT-milk)
1,0.052466,(beef)
2,0.033249,(berries)
3,0.026029,(beverages)
4,0.080529,(bottled beer)


In [ ]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    frequent_items,
    metric='confidence',
    min_threshold=0.3
)

print(rules.head())

       antecedents         consequents  antecedent support  \
0           (beef)        (whole milk)            0.052466   
1  (bottled water)        (whole milk)            0.110524   
2    (brown bread)        (whole milk)            0.064870   
3         (butter)  (other vegetables)            0.055414   
4         (butter)        (whole milk)            0.055414   

   consequent support   support  confidence      lift  representativity  \
0            0.255516  0.021251    0.405039  1.585180               1.0   
1            0.255516  0.034367    0.310948  1.216940               1.0   
2            0.255516  0.025216    0.388715  1.521293               1.0   
3            0.193493  0.020031    0.361468  1.868122               1.0   
4            0.255516  0.027555    0.497248  1.946053               1.0   

   leverage  conviction  zhangs_metric   jaccard  certainty  kulczynski  
0  0.007845    1.251315       0.389597  0.074113   0.200841    0.244103  
1  0.006126    1.080446     

## Understanding Column 0 Rule

### Rule
Beef → Whole Milk

### Columns Explained

| Column | Value | Meaning |
|----------|----------|----------|
| antecedents | beef | Product bought first |
| consequents | whole milk | Product associated with beef |
| antecedent support | 0.052466 | 5.25% of all transactions contain beef |
| consequent support | 0.255516 | 25.55% of all transactions contain whole milk |
| support | 0.021251 | 2.13% of all transactions contain both beef and whole milk |
| confidence | 0.405039 | 40.5% of beef buyers also buy whole milk |
| lift | 1.585180 | Beef buyers are 1.58 times more likely to buy whole milk |
| leverage | 0.007845 | Extra association above random chance |
| conviction | 1.251315 | Strength of implication (higher = stronger rule) |
| zhangs_metric | 0.389597 | Positive relationship between products |
| jaccard | 0.074113 | Similarity between beef and whole milk purchases |
| certainty | 0.200841 | Confidence improvement over random chance |
| kulczynski | 0.244103 | Average association strength between products |
| representativity | 1.0 | Rule fully represents the discovered pattern |

### Simple Interpretation

Out of all customers:

- 5.25% bought beef.
- 25.55% bought whole milk.
- 2.13% bought both.
- Among beef buyers, 40.5% also bought whole milk.
- Beef buyers are 1.58× more likely to buy whole milk than an average customer.

**Conclusion:** Beef and whole milk have a positive purchasing relationship.

In [ ]:
rules.sort_values(
    by='lift',
    ascending=False
).reset_index(drop=True).head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,"(other vegetables, whole milk)",(root vegetables),0.074835,0.108998,0.023183,0.309783,2.842082,1.0,0.015026,1.290900,0.700572,0.144304,0.225347,0.261235
1,"(root vegetables, whole milk)",(other vegetables),0.048907,0.193493,0.023183,0.474012,2.449770,1.0,0.013719,1.533320,0.622230,0.105751,0.347821,0.296912
2,(root vegetables),(other vegetables),0.108998,0.193493,0.047382,0.434701,2.246605,1.0,0.026291,1.426693,0.622764,0.185731,0.299078,0.339789
3,(whipped/sour cream),(other vegetables),0.071683,0.193493,0.028876,0.402837,2.081924,1.0,0.015006,1.350565,0.559803,0.122203,0.259569,0.276037
4,"(whole milk, yogurt)",(other vegetables),0.056024,0.193493,0.022267,0.397459,2.054131,1.0,0.011427,1.338511,0.543633,0.097987,0.252901,0.256270
5,"(other vegetables, yogurt)",(whole milk),0.043416,0.255516,0.022267,0.512881,2.007235,1.0,0.011174,1.528340,0.524577,0.080485,0.345695,0.300014
6,(butter),(whole milk),0.055414,0.255516,0.027555,0.497248,1.946053,1.0,0.013395,1.480817,0.514659,0.097237,0.324697,0.302543
7,(pork),(other vegetables),0.057651,0.193493,0.021657,0.375661,1.941476,1.0,0.010502,1.291779,0.514595,0.094373,0.225874,0.243795
8,(curd),(whole milk),0.053279,0.255516,0.026131,0.490458,1.919481,1.0,0.012517,1.461085,0.505984,0.092446,0.315577,0.296363
9,"(root vegetables, other vegetables)",(whole milk),0.047382,0.255516,0.023183,0.489270,1.914833,1.0,0.011076,1.457687,0.501524,0.082879,0.313982,0.289999


### Summary of Top 5 Rules

1. **Other Vegetables + Whole Milk → Root Vegetables**
   - Strongest rule (Lift = 2.84).
   - Customers buying vegetables and milk are highly likely to also buy root vegetables.

2. **Root Vegetables + Whole Milk → Other Vegetables**
   - Strong association (Lift = 2.45).
   - Customers purchasing root vegetables and milk often buy other vegetables.

3. **Root Vegetables → Other Vegetables**
   - Strong relationship (Lift = 2.25).
   - Vegetable products are commonly purchased together.

4. **Whipped/Sour Cream → Other Vegetables**
   - Positive association (Lift = 2.08).
   - Customers buying whipped/sour cream frequently purchase vegetables.

5. **Whole Milk + Yogurt → Other Vegetables**
   - Strong relationship (Lift = 2.05).
   - Dairy buyers often purchase vegetables in the same transaction.

### Conclusion

The analysis shows that **Whole Milk**, **Root Vegetables**, and **Other Vegetables** are key products that frequently appear together in customer baskets. The strongest association found was between **Other Vegetables + Whole Milk** and **Root Vegetables** (Lift = 2.84), indicating a strong purchasing pattern among these items.

### Important Apriori Metrics

#### 1. Antecedent
**Definition:** Product(s) purchased first (left side of the rule).

**Example Rule:**
Beef → Whole Milk

**Real Life:**
Customer buys Beef.

---

#### 2. Consequent
**Definition:** Product(s) associated with the antecedent (right side of the rule).

**Example Rule:**
Beef → Whole Milk

**Real Life:**
Customer who buys Beef may also buy Whole Milk.

---

#### 3. Antecedent Support
**Definition:** Percentage of transactions containing the antecedent.

**Formula:**
Antecedent Support = Transactions with A / Total Transactions

**Example:**
Beef → Whole Milk

Antecedent Support = 0.052

**Real Life:**
About 5.2% of all customers bought Beef.

---

#### 4. Consequent Support
**Definition:** Percentage of transactions containing the consequent.

**Formula:**
Consequent Support = Transactions with B / Total Transactions

**Example:**
Whole Milk Support = 0.255

**Real Life:**
About 25.5% of all customers bought Whole Milk.

---

#### 5. Support
**Definition:** Percentage of transactions containing BOTH antecedent and consequent.

**Formula:**
Support = Transactions with A and B / Total Transactions

**Example:**
Beef + Whole Milk = 0.021

**Real Life:**
About 2.1% of all customers bought both Beef and Whole Milk.

---

#### 6. Confidence
**Definition:** Probability of buying B when A is purchased.

**Formula:**
Confidence = Support(A,B) / Support(A)

**Example:**
Beef → Whole Milk

Confidence = 0.405

**Real Life:**
Out of 100 customers who bought Beef, about 40 also bought Whole Milk.

---

#### 7. Lift
**Definition:** Strength of the relationship between products.

**Formula:**
Lift = Confidence / Support(B)

**Example:**
Beef → Whole Milk

Lift = 1.58

**Real Life:**
Customers who buy Beef are 1.58 times more likely to buy Whole Milk than a random customer.

Interpretation:
- Lift = 1 → No relationship
- Lift > 1 → Positive relationship
- Lift < 1 → Negative relationship

So:

Count = 4

means:

4 different shopping baskets contained Milk.

not

4 milk packets were sold.